In [0]:
select * from src_ath3d

In [0]:
-- 1. JDBC SQL Server Connection
--https://community.cloud.databricks.com/#secrets/createScope
--databricks secrets put-secret dev-secret-etl vp-dev-host --string-value "sampl-sql-ash.database.windows.net"
--databricks secrets put-secret dev-secret-etl vp-dev-port --string-value "1433"
--databricks secrets put-secret dev-secret-etl vp-dev-username --string-value "ash"
--databricks secrets put-secret dev-secret-etl vp-dev-password --string-value "Password1!"
--databricks secrets put-secret dev-secret-etl vp-dev-db-name --string-value "sql-db"
INSERT INTO etl_metadata.etl_connection_config (
    connection_id,
    connection_name,
    connection_type,
    host_kv_secret,
    port_kv_secret,
    username_kv_secret,
    password_kv_secret,
    database_kv_secret,
    jdbc_driver_class,
    extra_options_json,
    is_active,
    created_by,
    updated_by
) VALUES (
    'CONN_MSSQL_VISIONPLUS',
    'VisionPlus Production Database',
    'JDBC',
    'vp-dev-host',           -- Name of secret in AKV
    'vp-dev-port',           -- Name of secret in AKV
    'vp-dev-username',       -- Name of secret in AKV
    'vp-dev-password',       -- Name of secret in AKV
    'vp-dev-db-name',        -- Name of secret in AKV
    'com.microsoft.sqlserver.jdbc.SQLServerDriver',
    '{"encrypt": "true", "trustServerCertificate": "false", "loginTimeout": "30"}',
    true,
    'admin_user',
    'admin_user'
);

-- 2. ADLS Gen2 Storage Connection
INSERT INTO etl_metadata.etl_connection_config (
    connection_id,
    connection_name,
    connection_type,
    host_kv_secret,           -- Used for the Storage Account Name or Endpoint
    password_kv_secret,       -- Used for the Service Principal Client Secret or Account Key
    extra_options_json,
    is_active,
    created_by,
    updated_by
) VALUES (
    'CONN_ADLS_LANDING',
    'Azure Data Lake Landing Zone',
    'ADLS',
    'adls-landing-account-name',
    'adls-landing-client-secret',
    '{"container": "raw", "tenant_id_secret": "adls-tenant-id", "client_id_secret": "adls-client-id"}',
    true,
    'admin_user',
    'admin_user'
);

-- 3. SFTP External Vendor Connection
INSERT INTO etl_metadata.etl_connection_config (
    connection_id,
    connection_name,
    connection_type,
    host_kv_secret,
    port_kv_secret,
    username_kv_secret,
    password_kv_secret,
    is_active,
    created_by,
    updated_by
) VALUES (
    'CONN_SFTP_VENDOR_A',
    'External Vendor A SFTP',
    'SFTP',
    'vendor-a-sftp-url',
    'vendor-a-sftp-port',
    'vendor-a-sftp-username',
    'vendor-a-sftp-password',
    true,
    'admin_user',
    'admin_user'
);

In [0]:
%python
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from datetime import datetime, date
from typing import Optional, List, Tuple
import logging

jobs = [
    Row(job_id=1, execution_order=10),
    Row(job_id=2, execution_order=20),
    Row(job_id=3, execution_order=20),
    Row(job_id=4, execution_order=30),
]

def group_jobs_by_execution_order(
        
        jobs: List[Row]
    ) -> List[List[Row]]:
        """
        Groups job_config rows into execution waves.
        Jobs in the same wave (same execution_order) run in parallel.
        Waves are returned in ascending order.

        Example:
            Wave 1 (order=1): [job_A, job_B, job_C]  ← parallel
            Wave 2 (order=2): [job_D]                 ← sequential (depends on wave 1)
        """
        from collections import defaultdict
        waves: dict = defaultdict(list)
        for job in jobs:
            waves[job["execution_order"]].append(job)
        print(waves)
        return [waves[k] for k in sorted(waves.keys())]
print(group_jobs_by_execution_order(jobs))


In [0]:
%python
import sys
import os
import logging
from datetime import date

# -- Make framework modules importable ---------------------------------------
# In Databricks, the repo is mounted at /Workspace/Repos/<user>/etl_framework
REPO_ROOT = "/Workspace/Users/ashok.k.gupta121@gmail.com/etl_framework"
MODULES_PATH = f"{REPO_ROOT}/framework/modules"
SQL_PATH = f"{REPO_ROOT}/framework/modules/sql/visionplus"


# DBTITLE 1,Importing Required Libraries
import pandas as pd
import numpy as np


if MODULES_PATH not in sys.path:
    sys.path.insert(0, MODULES_PATH)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
if SQL_PATH not in sys.path:
    print(SQL_PATH)
    sys.path.insert(0, SQL_PATH)

# -- Validate path is correct before importing --------------------------------
import importlib, importlib.util


from brz_visionplus_ath2_lrpmt_tgt import BrzVisionplusAth2LrpmtTgt
from brz_visionplus_ath3d_tgt import   BrzVisionplusAth3dTgt
from brz_visionplus_ath3x_tgt import BrzVisionplusAth3xTgt
from brz_visionplus_atptrpt_tgt import BrzVisionplusAtptrptTgt


class_obj = globals()['BrzVisionplusAth3xTgt']({'watermark_value': '2026-03-01', 'business_date': '2026-02-01'})
print(class_obj.business_date)
queries = getattr(class_obj, "queries", None)
print(queries)

In [0]:
update etl_metadata.job_config
set batch_job_config_id =1,notebook_name ='BrzVisionplusAth2LrpmtTgt',table_strategy="FULL",
execution_order = 1,depends_on_job_ids = null
where job_config_id = 1;

update etl_metadata.job_config
set batch_job_config_id =1,notebook_name ='BrzVisionplusAtptrptTgt',table_strategy="APPEND",
execution_order = 1,depends_on_job_ids = null
where job_config_id = 2;

update etl_metadata.job_config
set batch_job_config_id =2,notebook_name ='BrzVisionplusAth3dTgt',
table_strategy="FULL",execution_order = 1,depends_on_job_ids = 1
where job_config_id = 3;

update etl_metadata.job_config
set batch_job_config_id =2,notebook_name ='BrzVisionplusAth3xTgt',
table_strategy="FULL",execution_order = 1,depends_on_job_ids = 2
where job_config_id = 4;

In [0]:
select * from etl_metadata.job_audit_log; --where  batch_job_config_id =1;
select * from etl_metadata.batch_job_log ;--where  batch_job_config_id =1;

In [0]:
DROP TABLE IF EXISTS src_ath3x;
CREATE TABLE IF NOT EXISTS src_ath3x (
    ACCT_ID            STRING,
    CYCLE_DT           DATE,
    STMT_DT            DATE,
    AUTH_AMT           DECIMAL(18,2),
    AUTH_TYPE_CD       STRING,
    MERCH_ID           STRING,
    MERCH_NM           STRING,
    MERCH_CITY         STRING,
    MERCH_COUNTRY_CD   STRING,
    TERMINAL_ID        STRING,
    POS_ENTRY_CD       STRING,
    DECLINE_RSN_CD     STRING,
    CREATE_DT          TIMESTAMP,
    UPDATE_DT          TIMESTAMP,
    business_date      DATE
)
USING DELTA;



In [0]:
drop table if exists default.src_atptrpt;
CREATE TABLE default.src_atptrpt (
    ACCT_ID STRING,
    RPT_DT DATE,
    RPT_TYPE_CD STRING,
    RPT_STATUS_CD STRING,
    RPT_AMT DECIMAL(18,2),
    CURR_CD STRING,
    POSTING_DT DATE,
    REVERSAL_IND STRING,
    ORIG_RPT_ID STRING,
    CREATE_DT TIMESTAMP,
    UPDATE_DT TIMESTAMP,
    business_date DATE
)
USING DELTA;

In [0]:
drop table if exists default.src_ath3d;
CREATE TABLE IF NOT EXISTS src_ath3d (
    ACCT_ID                STRING,
    ACCT_TYPE_CD           STRING,
    ACCT_STATUS_CD         STRING,
    OPEN_DT                DATE,
    CLOSE_DT               DATE,
    CREDIT_LIMIT           DECIMAL(18,2),
    CURR_BALANCE           DECIMAL(18,2),
    AVAIL_CREDIT           DECIMAL(18,2),
    STMT_CYCLE_DY          INT,
    PMT_DUE_DY             INT,
    INTEREST_RATE          DECIMAL(5,2),
    CURR_CD                STRING,
    BRANCH_CD              STRING,
    RELATIONSHIP_MGR_ID    STRING,
    CREATE_DT              TIMESTAMP,
    UPDATE_DT              TIMESTAMP,
    business_date          DATE
)
USING DELTA;



In [0]:
drop table if exists default.src_ath2_lrpmt;
CREATE TABLE IF NOT EXISTS default.src_ath2_lrpmt (
    ACCT_ID STRING,
    SEQ_NO STRING,
    PMT_DT DATE,
    PMT_AMT DOUBLE,
    PMT_TYPE_CD STRING,
    PMT_STATUS_CD STRING,
    ORIG_PMT_AMT DOUBLE,
    CURR_CD STRING,
    POST_DT DATE,
    EFFECTIVE_DT DATE,
    CREATE_DT TIMESTAMP,
    UPDATE_DT TIMESTAMP,
    LOAD_DT TIMESTAMP,
    business_date DATE
)





In [0]:
CREATE TABLE IF NOT EXISTS etl_metadata.batch_job_log (
    batch_job_log_id        STRING          NOT NULL    COMMENT 'Composite PK: {batch_job_config_id}_{YYYYMMDDHHMMSS}',
    batch_job_config_id     BIGINT          NOT NULL    COMMENT 'FK → batch_job_config.batch_job_config_id',
    business_date           DATE            NOT NULL    COMMENT 'Logical processing/business date',
    batch_start_date        TIMESTAMP                   COMMENT 'Actual wall-clock batch start time',
    batch_end_date          TIMESTAMP                   COMMENT 'Actual wall-clock batch end time',
    batch_status            STRING          NOT NULL    COMMENT 'RUNNING | COMPLETED | FAILED | SKIPPED',
    batch_airflow_dag_id    STRING                      COMMENT 'Airflow DAG run ID for traceability',
    skip_reason             STRING          DEFAULT 0            COMMENT 'Populated when batch_status = SKIPPED',
    total_jobs              INT             DEFAULT 0   COMMENT 'Total tables in this batch run',
    jobs_completed          INT             DEFAULT 0,
    jobs_failed             INT             DEFAULT 0,
    jobs_skipped            INT             DEFAULT 0,
    triggered_by            STRING                      COMMENT 'SCHEDULED | MANUAL | RERUN | API',
    databricks_run_id       STRING                      COMMENT 'Databricks workflow run ID',
    error_message           STRING                      COMMENT 'Top-level error message if batch failed',
    created_dt              TIMESTAMP       DEFAULT current_timestamp(),
    updated_dt              TIMESTAMP       DEFAULT current_timestamp(),
    CONSTRAINT pk_batch_job_log PRIMARY KEY (batch_job_log_id)
)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'LOG TABLE 1 of 2 — Batch-level runtime audit log'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.logRetentionDuration'       = 'interval 180 days',
    'delta.feature.allowColumnDefaults' = 'supported'
)
;


In [0]:
-- =============================================================================
-- FILE    : 01_metadata_tables_ddl.sql
-- PURPOSE : DDL for all ETL framework metadata tables
--           3 config tables (one-time setup) + 2 runtime logging tables
-- SCHEMA  : etl_metadata
-- =============================================================================

CREATE SCHEMA IF NOT EXISTS etl_metadata
COMMENT 'ETL Framework control and logging schema';

create schema cdm_visionplus
COMMENT 'Target Schema';

In [0]:
-- =============================================================================
-- CONFIG TABLE 1: batch_job_config
-- PURPOSE : Defines a logical batch (group of tables) and its Airflow schedule.
--           One batch = one Airflow DAG.
-- POPULATED: One-time setup / when a new source system onboarded.
-- =============================================================================

CREATE TABLE IF NOT EXISTS etl_metadata.batch_job_config (
    batch_job_config_id     BIGINT          NOT NULL    COMMENT 'Surrogate PK, auto-incremented',
    source_system           STRING          NOT NULL    COMMENT 'Source system name (e.g. VisionPlus, SAP, Salesforce)',
    batch_name              STRING          NOT NULL    COMMENT 'Short batch identifier used in DAG naming (e.g. VP-1)',
    batch_desc              STRING                      COMMENT 'Business description of what this batch loads',
    layer                   STRING          NOT NULL    COMMENT 'Target layer transition: BRONZE | SILVER | GOLD | EL_TO_AEL',
    frequency               STRING          NOT NULL    COMMENT 'Schedule frequency: DAILY | WEEKLY | MONTHLY | ADHOC',
    airflow_dag_name        STRING          NOT NULL    COMMENT 'Airflow DAG ID that drives this batch',
    schedule_cron           STRING                      COMMENT 'Cron expression (e.g. 0 2 * * *); NULL = triggered externally',
    schedule_timezone       STRING          DEFAULT 'UTC' COMMENT 'Timezone for schedule (e.g. Asia/Kolkata)',
    max_parallel_jobs       INT             DEFAULT 5   COMMENT 'Max tables to execute in parallel within this batch',
    retry_attempts          INT             DEFAULT 2   COMMENT 'Auto-retry count on failure',
    retry_delay_seconds     INT             DEFAULT 300 COMMENT 'Seconds to wait between retries',
    alert_email             STRING                      COMMENT 'Comma-separated alert email addresses',
    is_active               BOOLEAN         DEFAULT TRUE COMMENT 'Soft-disable flag; FALSE = batch skipped by orchestrator',
    created_by              STRING                      COMMENT 'User who created this config record',
    created_dt              TIMESTAMP       DEFAULT current_timestamp(),
    updated_by              STRING,
    updated_dt              TIMESTAMP       DEFAULT current_timestamp(),
    CONSTRAINT pk_batch_job_config PRIMARY KEY (batch_job_config_id)
)
USING DELTA
COMMENT 'CONFIG TABLE 1 of 3 — Batch-level config: one row per logical batch / Airflow DAG'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true','delta.feature.allowColumnDefaults' = 'supported')


;
-- =============================================================================
-- CONFIG TABLE 2: job_config
-- PURPOSE : Table-level ETL configuration within a batch.
--           Defines source→target mapping, load strategy, notebook, ordering.
-- POPULATED: One-time setup / when a new table is onboarded.
-- =============================================================================
CREATE TABLE IF NOT EXISTS etl_metadata.job_config (
    job_config_id           BIGINT          NOT NULL    COMMENT 'Surrogate PK, auto-incremented',
    batch_job_config_id     BIGINT          NOT NULL    COMMENT 'FK → batch_job_config.batch_job_config_id',

    -- Source definition
    source_system           STRING                      COMMENT 'Overrides batch-level source_system if different',
    source_schema           STRING                      COMMENT 'Source schema/database name',
    source_table_name       STRING                      COMMENT 'Source table name (NULL if fully custom SQL)',
    source_watermark_column STRING                      COMMENT 'Column used for incremental high-watermark (e.g. updated_at)',
    source_watermark_value  STRING                      COMMENT 'Last loaded watermark value (updated after each run)',
    connection_id           STRING                      COMMENT 'FK → etl_connection_config.connection_id',

    -- Target definition
    target_schema           STRING          NOT NULL    COMMENT 'Target Delta schema (e.g. cdm_visionplus)',
    target_table_name       STRING          NOT NULL    COMMENT 'Target Delta table name (e.g. ATH2_LRPMT_TGT)',
    primary_key_columns     STRING                      COMMENT 'Comma-separated PK columns for MERGE/SCD logic',
    partition_columns       STRING                      COMMENT 'Comma-separated columns for Delta PARTITIONED BY',

    -- Transformation
    notebook_name           STRING          NOT NULL    COMMENT 'Transformation notebook path relative to sql/ directory',
    target_frequency        STRING          NOT NULL    COMMENT 'DAILY | WEEKLY | MONTHLY | ADHOC',
    table_strategy          STRING          NOT NULL    COMMENT 'FULL | INC (incremental) | SCD1 | SCD2 | APPEND',

    -- Dependency & ordering
    depends_on_job_ids      STRING                      COMMENT 'Comma-separated job_config_ids this job depends on',
    execution_order         INT             DEFAULT 1   COMMENT 'Execution order within batch; same value = run in parallel',

    -- Quality & hooks
    expected_min_row_count  BIGINT                      COMMENT 'DQ check: minimum expected row count post-load',
    pre_hook_notebook       STRING                      COMMENT 'Optional notebook to run before this table load',
    post_hook_notebook      STRING                      COMMENT 'Optional notebook to run after this table load',

    is_active               BOOLEAN         DEFAULT TRUE COMMENT 'FALSE = skip this table during batch execution',
    created_by              STRING,
    created_dt              TIMESTAMP       DEFAULT current_timestamp(),
    updated_by              STRING,
    updated_dt              TIMESTAMP       DEFAULT current_timestamp(),
    CONSTRAINT pk_job_config PRIMARY KEY (job_config_id)
)
USING DELTA
COMMENT 'CONFIG TABLE 2 of 3 — Table-level ETL config: source/target mapping, strategy, notebook, ordering'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true','delta.feature.allowColumnDefaults' = 'supported')
;


-- =============================================================================
-- CONFIG TABLE 3: etl_connection_config
-- PURPOSE : Source system connection registry.
--           Actual secrets stored in Azure Key Vault; only KV secret names here.
-- POPULATED: One-time setup per source system.
-- =============================================================================
CREATE TABLE IF NOT EXISTS etl_metadata.etl_connection_config (
    connection_id               STRING      NOT NULL    COMMENT 'Unique ID (e.g. CONN_VP_PROD)',
    connection_name             STRING      NOT NULL    COMMENT 'Display name',
    connection_type             STRING      NOT NULL    COMMENT 'JDBC | SFTP | ADLS | REST_API | KAFKA | DELTA',
    host_kv_secret              STRING                  COMMENT 'Key Vault secret name for host/URL',
    port_kv_secret              STRING                  COMMENT 'Key Vault secret name for port',
    username_kv_secret          STRING                  COMMENT 'Key Vault secret name for username',
    password_kv_secret          STRING                  COMMENT 'Key Vault secret name for password',
    database_kv_secret          STRING                  COMMENT 'Key Vault secret name for database name',
    jdbc_driver_class           STRING                  COMMENT 'JDBC driver class (e.g. com.microsoft.sqlserver.jdbc.SQLServerDriver)',
    extra_options_json          STRING                  COMMENT 'JSON of additional JDBC/connection properties',
    is_active                   BOOLEAN     DEFAULT TRUE,
    created_by                  STRING,
    created_dt                  TIMESTAMP   DEFAULT current_timestamp(),
    updated_by                  STRING,
    updated_dt                  TIMESTAMP   DEFAULT current_timestamp(),
    CONSTRAINT pk_etl_connection_config PRIMARY KEY (connection_id)
)
USING DELTA
COMMENT 'CONFIG TABLE 3 of 3 — Source connection registry (secrets via Azure Key Vault)'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true','delta.feature.allowColumnDefaults' = 'supported');



-- =============================================================================
-- LOG TABLE 1: batch_job_log
-- PURPOSE : One row per batch execution (per business_date + DAG run).
--           Written by orchestrator at batch start; updated at batch end.
-- =============================================================================
CREATE TABLE IF NOT EXISTS etl_metadata.batch_job_log (
    batch_job_log_id        STRING          NOT NULL    COMMENT 'Composite PK: {batch_job_config_id}_{YYYYMMDDHHMMSS}',
    batch_job_config_id     BIGINT          NOT NULL    COMMENT 'FK → batch_job_config.batch_job_config_id',
    business_date           DATE            NOT NULL    COMMENT 'Logical processing/business date',
    batch_start_date        TIMESTAMP                   COMMENT 'Actual wall-clock batch start time',
    batch_end_date          TIMESTAMP                   COMMENT 'Actual wall-clock batch end time',
    batch_status            STRING          NOT NULL    COMMENT 'RUNNING | COMPLETED | FAILED | SKIPPED',
    batch_airflow_dag_id    STRING                      COMMENT 'Airflow DAG run ID for traceability',
    skip_reason             STRING          DEFAULT 0            COMMENT 'Populated when batch_status = SKIPPED',
    total_jobs              INT             DEFAULT 0   COMMENT 'Total tables in this batch run',
    jobs_completed          INT             DEFAULT 0,
    jobs_failed             INT             DEFAULT 0,
    jobs_skipped            INT             DEFAULT 0,
    triggered_by            STRING                      COMMENT 'SCHEDULED | MANUAL | RERUN | API',
    databricks_run_id       STRING                      COMMENT 'Databricks workflow run ID',
    error_message           STRING                      COMMENT 'Top-level error message if batch failed',
    created_dt              TIMESTAMP       DEFAULT current_timestamp(),
    updated_dt              TIMESTAMP       DEFAULT current_timestamp(),
    CONSTRAINT pk_batch_job_log PRIMARY KEY (batch_job_log_id)
)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'LOG TABLE 1 of 2 — Batch-level runtime audit log'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.logRetentionDuration'       = 'interval 180 days',
    'delta.feature.allowColumnDefaults' = 'supported'
)
;



-- =============================================================================
-- LOG TABLE 2: job_audit_log
-- PURPOSE : One row per individual table load within a batch run.
--           Written at table load start; updated at completion/failure.
-- =============================================================================
CREATE TABLE IF NOT EXISTS etl_metadata.job_audit_log (
    job_audit_log_id        BIGINT          NOT NULL    COMMENT 'Surrogate PK: generated as YYYYMMDDHHMMSSmmm + seq',
    business_date           DATE            NOT NULL    COMMENT 'Logical processing/business date',
    job_config_id           BIGINT          NOT NULL    COMMENT 'FK → job_config.job_config_id',
    batch_job_config_id     BIGINT          NOT NULL    COMMENT 'FK → batch_job_config.batch_job_config_id',
    batch_airflow_dag_id    STRING                      COMMENT 'FK → batch_job_log.batch_airflow_dag_id',
    batch_job_log_id        STRING                      COMMENT 'FK → batch_job_log.batch_job_log_id',
    job_status              STRING          NOT NULL    COMMENT 'RUNNING | COMPLETED | FAILED | SKIPPED',
    skip_reason             STRING                      COMMENT 'Reason when job_status = SKIPPED',
    job_start_date          TIMESTAMP                   COMMENT 'Table load start time',
    job_end_date            TIMESTAMP                   COMMENT 'Table load end time',
    duration_seconds        INT                         COMMENT 'Computed: job_end_date - job_start_date in seconds',
    source_row_count        BIGINT                      COMMENT 'Rows read from source',
    target_rows_inserted    BIGINT                      COMMENT 'Net new rows inserted into target',
    target_rows_updated     BIGINT                      COMMENT 'Rows updated/merged in target',
    target_rows_deleted     BIGINT                      COMMENT 'Rows deleted (SCD2 / hard deletes)',
    target_total_row_count  BIGINT                      COMMENT 'Total rows in target table after load',
    watermark_value_used    STRING                      COMMENT 'Watermark value at start of this run',
    watermark_value_new     STRING                      COMMENT 'New high watermark after successful load',
    error_desc              STRING                      COMMENT 'Error description on failure; "Successful Run" on success',
    notebook_path           STRING                      COMMENT 'Actual notebook path executed',
    retry_attempt           INT             DEFAULT 0   COMMENT '0 = first attempt; 1,2 = retries',
    databricks_task_run_id  STRING                      COMMENT 'Databricks task-level run ID',
    created_dt              TIMESTAMP       DEFAULT current_timestamp(),
    updated_dt              TIMESTAMP       DEFAULT current_timestamp(),
    CONSTRAINT pk_job_audit_log PRIMARY KEY (job_audit_log_id)
)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'LOG TABLE 2 of 2 — Table-level runtime audit log with row counts and watermarks'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.logRetentionDuration'       = 'interval 180 days',
    'delta.feature.allowColumnDefaults' = 'supported'
)
;


-- =============================================================================
-- SAMPLE CONFIG INSERT (seed data to match xlsx examples)
-- =============================================================================

INSERT INTO etl_metadata.batch_job_config
    (batch_job_config_id, source_system, batch_name, batch_desc, layer, frequency, airflow_dag_name, is_active, created_by)
VALUES
    (1, 'VisionPlus', 'VP-1', 'First batch of VP',  'EL_TO_AEL', 'Daily', 'dag_visionplus_vp1', TRUE, 'framework_init'),
    (2, 'VisionPlus', 'VP-2', 'Second batch of VP', 'EL_TO_AEL', 'Daily', 'dag_visionplus_vp2', TRUE, 'framework_init');

INSERT INTO etl_metadata.job_config
    (job_config_id, batch_job_config_id, target_schema, target_table_name, notebook_name, target_frequency, table_strategy, execution_order, is_active, created_by)
VALUES
    (1, 1, 'cdm_visionplus', 'ATH2_LRPMT_TGT',  '/Workspace/Users/ashok.k.gupta121@gmail.com/etl_framework/framework/sql/bronze/visionplus/brz_visionplus_ath2_lrpmt_tgt',  'DAILY', 'FULL', 1, TRUE, 'framework_init'),
    (2, 1, 'cdm_visionplus', 'ATPTRPT_TGT',      'bronze/visionplus/brz_visionplus_atptrpt',     'DAILY', 'INC',  1, TRUE, 'framework_init'),
    (3, 1, 'cdm_visionplus', 'ATH3X_TGT',        'bronze/visionplus/brz_visionplus_ath3x',       'DAILY', 'INC',  1, TRUE, 'framework_init'),
    (4, 1, 'cdm_visionplus', 'ATH3D_TGT',        'bronze/visionplus/brz_visionplus_ath3d',       'DAILY', 'FULL', 2, TRUE, 'framework_init');
